In [1]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from scipy.stats import zscore
from sklearn.utils import resample

In [2]:
titanic = fetch_openml(name='titanic', version=1, as_frame=True)
df = titanic.frame

print(df.head())

   pclass survived                                             name     sex  \
0       1        1                    Allen, Miss. Elisabeth Walton  female   
1       1        1                   Allison, Master. Hudson Trevor    male   
2       1        0                     Allison, Miss. Helen Loraine  female   
3       1        0             Allison, Mr. Hudson Joshua Creighton    male   
4       1        0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   

       age  sibsp  parch  ticket      fare    cabin embarked boat   body  \
0  29.0000      0      0   24160  211.3375       B5        S    2    NaN   
1   0.9167      1      2  113781  151.5500  C22 C26        S   11    NaN   
2   2.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3  30.0000      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4  25.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest  
0                     St Louis,

In [3]:
num_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')

df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [4]:
scale_cols = ['age', 'fare']

standard = StandardScaler()
df[scale_cols] = standard.fit_transform(df[scale_cols])

In [5]:
z = abs(zscore(df[scale_cols]))
df = df[(z < 3).all(axis=1)]

In [7]:
df = pd.get_dummies(df, columns=['sex','embarked'])

In [8]:
majority = df[df['survived'] == '0']
minority = df[df['survived'] == '1']

minority_upsampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

balanced_df = pd.concat([majority, minority_upsampled])

print(balanced_df['survived'].value_counts())

survived
0    791
1    791
Name: count, dtype: int64
